In [1]:
import os
os.chdir("..")


In [2]:
import pandas as pd
from imap_tools import AND, UidRange
from tqdm import trange, tqdm
from collections import defaultdict
import socket
import ssl
import os
from src.imap.IMAPClientOperations import IMAPClientOperations as IMAPClient

In [3]:
EMAIL = "info@geekprom.ru"
BATCH_SIZE = 20
conversations = defaultdict(lambda: {'original': None, 'reply': None})

In [10]:
client = IMAPClient().client

[2025-08-26 16:20:04.157743][IMAP][INFO] Connected to imap.yandex.ru:993
[2025-08-26 16:20:04.353112][IMAP][INFO] Login successfully


In [11]:
import pandas as pd
from imap_tools import AND
from tqdm import tqdm
import csv
import os

fieldnames = [
            'uid', 'subject', 'from', 'in_reply_to', 'to', 'date', 'text', 'html',
            'flags', 'size', 'attachments_count', 'headers'
        ]
fieldnames_attachments = [
            'uid', 'type', 'dispose', 'filename'
        ]



def imap_to_csv(folder: str = 'INBOX',
                csv_filename: str = 'emails.csv',
                batch_size: int = 100):
    data = pd.DataFrame(columns=fieldnames)
    try:
        total_emails = int(client.folder.set(folder)[1][0])
        print(f"Найдено писем: {total_emails}")
        
        with tqdm(total=total_emails, desc="Обработка писем", unit="письмо") as pbar:
            
            batch_count = 0
            emails_processed = 0
            
            # Обрабатываем письма батчами
            for msg in client.fetch(AND(all=True), bulk=10, mark_seen=False):
                email_data = {
                    'uid': msg.uid,
                    'subject': msg.subject or '',
                    'from': msg.from_ or '',
                    'in_reply_to': msg.reply_to or '',
                    'to': ', '.join(msg.to) if msg.to else '',
                    'date': msg.date.strftime('%Y-%m-%d %H:%M:%S') if msg.date else '',
                    'text': msg.text or '',
                    'html': msg.html or '',
                    'flags': ', '.join(msg.flags) if msg.flags else '',
                    'size': msg.size,
                    'attachments_count': len(msg.attachments),
                    'headers': str(msg.headers) if msg.headers else ''
                }
                
                # Записываем данные
                data = pd.concat([data, pd.DataFrame([email_data])], ignore_index=True)
                
                emails_processed += 1
                batch_count += 1
                pbar.update(1)
                
                # Обновляем прогресс каждые batch_size писем
                if batch_count >= batch_size:
                    data.to_csv(csv_filename)
                    pbar.set_postfix({
                        'обработано': f'{emails_processed}/{total_emails}',
                        'батч': batch_count
                    })

                    batch_count = 0
        data.to_csv(csv_filename)
        print(f"Данные сохранены в файл: {csv_filename}")
        
    except Exception as e:
        print(f"Ошибка при обработке писем: {e}")

In [12]:
def imap_to_csv_attachments(folder: str = 'INBOX',
                csv_filename: str = 'emails.csv',
                batch_size: int = 100):
    data = pd.DataFrame(columns=fieldnames_attachments)
    try:
        total_emails = int(client.folder.set(folder)[1][0])
        print(f"Найдено писем: {total_emails}")
        
        with tqdm(total=total_emails, desc="Обработка писем", unit="письмо") as pbar:
            
            batch_count = 0
            emails_processed = 0
            
            # Обрабатываем письма батчами
            for msg in client.fetch(AND(all=True), bulk=10, mark_seen=False):
                for num, attach in enumerate(msg.attachments):
                    if attach.content_type == 'inline':
                        continue
                    filename = f"{msg.uid}_{num}"
                    email_data = {
                        'uid': msg.uid,
                        'type': attach.content_type,
                        'dispose': attach.content_disposition,
                        'filename': filename
                    }
                    with open(f'./data_collecting/att/{filename}', 'wb') as f:
                        f.write(attach.payload)
                    data = pd.concat([data, pd.DataFrame([email_data])], ignore_index=True)
                
                emails_processed += 1
                batch_count += 1
                pbar.update(1)
                
                if batch_count >= batch_size:
                    data.to_csv(csv_filename)
                    pbar.set_postfix({
                        'обработано': f'{emails_processed}/{total_emails}',
                        'батч': batch_count
                    })

                    batch_count = 0
        data.to_csv(csv_filename)
        print(f"Данные сохранены в файл: {csv_filename}")
        
    except Exception as e:
        print(f"Ошибка при обработке писем: {e}")

In [13]:

print("="*50)
print(f"Старт обработки почты {EMAIL}")
print("="*50)

imap_to_csv_attachments(folder='ОТВЕЧЕНО', csv_filename='./data_collecting/answered_attachments.csv')
# imap_to_csv(folder='Sent', csv_filename='./data_collecting/sent.csv')



Старт обработки почты info@geekprom.ru
Найдено писем: 31743


Обработка писем: 31749письмо [51:47, 10.22письмо/s, обработано=31700/31743, батч=100]                         

Данные сохранены в файл: ./data_collecting/answered_attachments.csv


In [4]:
data_answered = pd.read_csv("./data_collecting/answered.csv", low_memory=False)
data_sent = pd.read_csv("./data_collecting/sent.csv", low_memory=False)

In [4]:
data_sent[data_sent['in_reply_to'].notna()]

,uid,subject,from,in_reply_to,to,date,text,html,flags,size,attachments_count,headers
344,43061,Re: Запрос цены,le@eco-intech.com,"('le@eco-intech.com',)",info@geekprom.ru,2025-07-11 13:09:22,NaN,<html><head><title>Re: &#1047;&#1072;&#1087;&#...,"\Seen, encrypted, system_hamon",10613,0,{'received': ('from postback27b.mail.yandex.ne...
360,43923,Новый заказ #101635,info@geekprom.ru,"('Nabiullinaan@nknh.sibur.ru',)",info@geekprom.ru,2025-04-11 11:28:39,NaN,<!DOCTYPE html>\r\n<html>\r\n <head>\r\n ...,"\Seen, encrypted",11169,0,{'received': ('from postback6d.mail.yandex.net...
362,43925,Новый заказ #101636,info@geekprom.ru,"('d.mytarev@saratovmeteo.san.ru',)",info@geekprom.ru,2025-04-11 11:44:58,NaN,<!DOCTYPE html>\r\n<html>\r\n <head>\r\n ...,"\Seen, encrypted",10691,0,{'received': ('from postback29b.mail.yandex.ne...
364,43927,Новый заказ #101637,info@geekprom.ru,"('d.mytarev@saratovmeteo.san.ru',)",info@geekprom.ru,2025-04-11 11:48:02,NaN,<!DOCTYPE html>\r\n<html>\r\n <head>\r\n ...,"\Seen, encrypted",10705,0,{'received': ('from postback24b.mail.yandex.ne...
366,43929,Новый заказ #101638,info@geekprom.ru,"('k-97kat@yandex.ru',)",info@geekprom.ru,2025-04-11 12:10:21,NaN,<!DOCTYPE html>\r\n<html>\r\n <head>\r\n ...,"\Seen, encrypted",10520,0,{'received': ('from postback24d.mail.yandex.ne...
...,...,...,...,...,...,...,...,...,...,...,...,...
85654,90199,Новый заказ #102958,info@geekprom.ru,"('skv099@yandex.ru',)",info@geekprom.ru,2025-07-18 12:55:16,NaN,<!DOCTYPE html>\r\n<html>\r\n <head>\r\n ...,encrypted,10568,0,{'received': ('from postback29d.mail.yandex.ne...
85668,90213,Новый заказ #102959,info@geekprom.ru,"('tpk-42@mail.ru',)",info@geekprom.ru,2025-07-18 13:57:46,NaN,<!DOCTYPE html>\r\n<html>\r\n <head>\r\n ...,encrypted,9966,0,{'received': ('from postback19d.mail.yandex.ne...
85671,90216,Новый заказ #102960,info@geekprom.ru,"('m.zakharov@mail.ru',)",info@geekprom.ru,2025-07-18 14:18:09,NaN,<!DOCTYPE html>\r\n<html>\r\n <head>\r\n ...,encrypted,9460,0,{'received': ('from postback1d.mail.yandex.net...
85727,90278,Новый заказ #102402,info@geekprom.ru,"('inna@lightcom.su',)",info@geekprom.ru,2025-06-11 14:36:38,NaN,<!DOCTYPE html>\r\n<html>\r\n <head>\r\n ...,"\Seen, encrypted",9567,0,{'received': ('from postback7d.mail.yandex.net...


In [39]:
data_answered.headers.values[0]

'{\'received\': (\'from postback10b.mail.yandex.net (postback10b.mail.yandex.net [2a02:6b8:c02:900:1:45:d181:da10])\\r\\n\\tby d7ii3dnexkqmjmaz.myt.yp-c.yandex.net (notsolitesrv/Yandex) with LMTP id oq3NJlhasdgO-MPU0xPnH;\\r\\n\\tWed, 25 Oct 2023 11:37:26 +0300\', \'from mail-nwsmtp-mxback-production-main-42.iva.yp-c.yandex.net (mail-nwsmtp-mxback-production-main-42.iva.yp-c.yandex.net [IPv6:2a02:6b8:c0c:891e:0:640:dc19:0])\\r\\n\\tby postback10b.mail.yandex.net (Yandex) with ESMTP id A164860918;\\r\\n\\tWed, 25 Oct 2023 11:37:26 +0300 (MSK)\', \'from mail.yandex.ru (2a02:6b8:c0c:98a3:0:640:5e48:0 [2a02:6b8:c0c:98a3:0:640:5e48:0])\\r\\n\\tby mail-nwsmtp-mxback-production-main-42.iva.yp-c.yandex.net (mxback/Yandex) with HTTP id PbKrJ31W5eA0-VZQNLGeR;\\r\\n\\tWed, 25 Oct 2023 11:37:26 +0300\', \'by 6mha55xjikky5vne.iva.yp-c.yandex.net with HTTP;\\r\\n\\tWed, 25 Oct 2023 11:37:25 +0300\'), \'x-yandex-fwd\': (\'1\',), \'dkim-signature\': (\'v=1; a=rsa-sha256; c=relaxed/relaxed; d=sbr-promi

In [60]:
import ast
from collections import defaultdict
import re

def extract_in_reply_to(headers_str):
    try:
        headers_dict = ast.literal_eval(headers_str)
        message_id = headers_dict['in-reply-to'][0]
        return message_id
    except Exception as e:
        return 'No_rep'
        
replies = data_sent['headers'].map(extract_in_reply_to)
replies = replies[replies != 'No_rep']
replies

0           <d229ab88fce543ad8cdbecc7f1ea8810@interrao.ru>
1                         <648451709190238@mail.yandex.ru>
2                         <728641709191899@mail.yandex.ru>
3              <af1f82cde5cb479a993b21e3c4f3bb11@bgkrb.ru>
4        <8565924f-88ba-7061-2e03-3879121bafe4@analitpr...
                               ...                        
49466    <7c2f08aa94934f90b43127abc26b9fae@otcpharmpro.ru>
49467    <20250822103001.NT3TXs2s@mail-nwsmtp-smtp-prod...
49468                     <572101755868209@mail.yandex.ru>
49469          <5a7d8a8b1dc340b78614deb0c6c09c9e@ngco.pro>
49470                <1755861831.730675107@f733.i.mail.ru>
Name: headers, Length: 37169, dtype: object

In [61]:
import pandas as pd
import ast
import re

def extract_message_id(headers_str):
    try:
        headers_dict = ast.literal_eval(headers_str)
        message_id = headers_dict['message-id'][0]
        return message_id
    except Exception as e:
        return 'No_msg_id'
ids = data_answered.headers.map(extract_message_id)
ids.value_counts()

headers
$null                                                                              9
<sender@slavyanka.com>                                                             2
<431531739516089@mail.yandex.ru>                                                   2
<c41918db777f4505ab35ab80743eba16@nornik.ru>                                       2
<85154611-6613-4C38-8623-BF7DB37AA179@icloud.com>                                  2
                                                                                  ..
<1750451729237270@mail.yandex.ru>                                                  1
<1729235547.701706459@f542.i.mail.ru>                                              1
<2894581729236219@mail.yandex.ru>                                                  1
<AM0P189MB06914F347AD61F02CFD90C4E8B402@AM0P189MB0691.EURP189.PROD.OUTLOOK.COM>    1
<057d01db2135$ee9bc1d0$cbd34570$@ellara.ru>                                        1
Name: count, Length: 31468, dtype: int64

In [6]:
client.folder.set('ОТВЕЧЕНО')

('OK', [b'31628'])

In [ ]:
msg = list(client.fetch(AND(uid='5918')))[0]

In [22]:
msg.attachments[0].payload

with open('./data_collecting/image1.jpg', 'wb') as f:
    f.write(msg.attachments[1].payload)

In [ ]:
pbar = tqdm(total=int(replies.shape[0]), desc='Oбработка')
conversations_count = 0

for em in ids:
    _count = 0
    if not em:
        continue
    for email in replies:
        if str(email) == str(em):
            _count += 1
            
    if _count>0:
        conversations_count+=1
        if _count>1:
            pass
            #print(f"UID {uid} has {_count} emails in headers")
    _count = 0
    pbar.update()
print(conversations_count)


Oбработка:  85%|████████▍ | 31483/37169 [04:14<00:45, 123.72it/s]


KeyboardInterrupt: 

In [62]:
data_sent['reply'] = data_sent.headers.map(extract_in_reply_to)
data_answered['msg_id'] = data_answered.headers.map(extract_message_id)

conversations = pd.merge(
    left=data_sent,
    right=data_answered,
    left_on='reply',
    right_on='msg_id',
    how='inner',
    suffixes=("_res", "_req")
)
conversations

,Unnamed: 0_res,uid_res,subject_res,from_res,in_reply_to_res,to_res,date_res,text_res,html_res,flags_res,...,in_reply_to_req,to_req,date_req,text_req,html_req,flags_req,size_req,attachments_count_req,headers_req,msg_id
0,41,7791,Re: упд 304,tender@geekprom.ru,NaN,kristina.trifonova@rusal.com,2024-07-04 11:29:00,NaN,"<div>У нас с вами обмен по ЭДО, выгруженный до...","\Seen, delayed_message, encrypted, system_hamo...",...,NaN,tender@geekprom.ru,2024-07-04 06:51:08,Добрый день!\r\n\r\nПришлите пожалуйста скан п...,"<html xmlns:v=""urn:schemas-microsoft-com:vml"" ...","\Seen, \Answered",110572,2,{'received': ('from mail-collectors-production...,<2756ec146bd14e7f91b7488c7916b0ee@rusal.com>
1,44,8066,Re: Акт сверки за 2 квартал и 1 полугодие 2024,tender@geekprom.ru,NaN,dolgushina@yargeo.novatek.ru,2024-07-10 13:22:47,NaN,"<div>Добрый день, Валентина Леонидовна!</div><...","\Seen, delayed_message, encrypted, system_hamo...",...,NaN,tender@geekprom.ru,2024-07-10 03:56:16,Доброе утро!\r\n\r\nНаправляю в Ваш адрес Акты...,"<html xmlns:v=""urn:schemas-microsoft-com:vml"" ...","\Seen, \Answered",272167,3,{'received': ('from mail-collectors-production...,<edc1c56805d34253a5a805262305a49d@yargeo.novat...
2,49,8750,Re: Заказ,info@geekprom.ru,NaN,gudkovaaa@layta.ru,2024-07-25 09:15:39,NaN,"<div>Альбина, доброго дня.</div><div>Счет с ук...","\Seen, delayed_message, encrypted, system_hamo...",...,NaN,info@geekprom.ru,2024-07-25 05:44:18,Добрый день!\r\nПро8шу счет и сроки:\r\nАналит...,"<html xmlns:v=""urn:schemas-microsoft-com:vml"" ...","\Seen, \Answered, encrypted, system_hamon",24476,0,{'received': ('from postback2a.mail.yandex.net...,<6844fa16f7b945688d37ec1c6fb3bb4d@srvsrt01mlp0...
3,124,13563,Re: Сообщение с сайта dv-expert.ru от tender@g...,tender@geekprom.ru,NaN,114@dv-expert.org,2024-10-07 11:58:05,NaN,<div>Добрый день!</div><div>нет не выиграли</d...,"\Seen, delayed_message, encrypted, undo_message",...,NaN,tender@geekprom.ru,2024-10-06 14:28:27,"Добрый день!\r\n\r\nПодскажите, так понимаю вы...","<html xmlns:v=""urn:schemas-microsoft-com:vml"" ...","\Seen, \Answered",46889,2,{'received': ('from mail-collectors-production...,<00d601db17e2$dd377ad0$97a67070$@dv-expert.org>
4,146,14543,Re: Акт сверки,tender@geekprom.ru,NaN,alekseeva@yargeo.novatek.ru,2024-10-17 16:19:53,NaN,<div>Акт сверки во вложении</div><div> </div><...,"\Seen, delayed_message, encrypted, system_hamo...",...,NaN,tender@geekprom.ru,2024-10-17 03:14:59,Доброе утро!\r\nПрошу подписать акт сверки.\r\...,"<html xmlns:v=""urn:schemas-microsoft-com:vml"" ...","\Seen, \Answered",254298,3,{'received': ('from mail-collectors-production...,<45cd46bd784d4fabae3b7670bb87cefa@yargeo.novat...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
23532,49462,93512,Re: Запрос Отисифарм Про - газоанализатор,info@geekprom.ru,NaN,askondratenko@otcpharmpro.ru,2025-08-22 15:40:36,NaN,"<div>Добрый день, от Вас поступила оплата по п...","\Seen, \Recent, delayed_message, encrypted, sy...",...,NaN,info@geekprom.ru,2025-07-29 14:25:23,"Игорь, добрый день!\r\n\r\nВ целях оформления ...","<html xmlns:v=""urn:schemas-microsoft-com:vml"" ...","\Seen, \Answered, encrypted",853839,4,{'received': ('from postback20a.mail.yandex.ne...,<1db901147e48416fa4bd9d828878f9ec@otcpharmpro.ru>
23533,49463,93514,Отгрузка,info@geekprom.ru,NaN,bazyaka_oleg@mail.ru,2025-08-22 16:06:38,NaN,"<div>Олег,</div><div><div style=""background-co...","\Seen, \Recent, delayed_message, encrypted, sy...",...,NaN,info@geekprom.ru,2025-08-22 15:30:22,"оплатил, когда ждать доставку?\r\n\r\n> 22 авг...","<html><head><meta http-equiv=""content-type"" co...","\Seen, \Answered, \Recent, encrypted, system_h...",40058,0,{'received': ('from postback7d.mail.yandex.net...,<301D3850-56DA-462F-919C-1B55CC349553@mail.ru>
23534,49468,93519,Re: Заявка,info@geekprom.ru,NaN,elektrobest@yandex.ru,2025-08-22 16:41:07,NaN,<div>Здравствуйте!</div><div> </div><div>Высыл...,"\Seen, \Recent, delayed_message, encrypted,

In [76]:
conv_uniq = conversations.drop_duplicates(subset=['msg_id']).drop_duplicates(subset=['date_req']).drop_duplicates(subset=['date_res'])

In [81]:
conv_uniq.text_req.value_counts()

text_req
\r\n                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                           

In [82]:
conv_uniq.to_csv('./data_collecting/conversations.csv')

In [40]:
print(list(folder.name for folder in client.folder.list()))

['ОТВЕЧЕНО', 'ОТВЕЧЕНО|Уведомления GEEKPROM', 'Поставщики', 'Поставщики|Политехформ-М', 'Поставщики|Промприбор-Р', 'Поставщики|Прочие поставщики', 'Поставщики|Пратон', 'Поставщики|Аналитприбор', 'Поставщики|Газ-Фармэк', 'Поставщики|ЭМИ-прибор', 'Поставщики|Румянцево СМР', 'Поставщики|Транспорт', 'Поставщики|Транспорт|Авто', 'Поставщики|Транспорт|Деловые Линии', 'Поставщики|Транспорт|СДЭК', 'Поставщики|Транспорт|ТК КИТ', 'Поставщики|Хромдет-Экология', 'АША', 'Банк', 'Банк|Альфа', 'Банк|Апельсин-Страхование', 'Банк|Контур.Диадок', 'Банк|Робокасса', 'Банк|ТочкаБанк', 'Бухгалтерия', 'Долгий ящик', 'Китай', 'ТЕНДЕРЫ', 'ТЕНДЕРЫ|Илья', 'ШЛЯПА', 'ШЛЯПА|АВТО-ШЛЯПА', 'ШЛЯПА|МИПСТРОЙ-Крост', 'Archive', 'Drafts', 'Drafts|template', 'INBOX', 'Outbox', 'Sent', 'Spam', 'Trash']


In [41]:
client.folder.set('Archive')

('OK', [b'2'])